# Predict and evaluate

Thin runner over `iiss.eval.batch`: every finished run on `valid`, `test` and `phone`. Each job exports the predictions of `weights/last.pt` (`runs/<run_id>/predictions/`) and scores them with pycocotools (`results/eval/<run_id>/<eval_set>/`).

- `phone` is also scored per capture condition from `metadata.csv` (`normal` and `low_light`). The classes with GT differ between conditions, so compare condition APs with care; the per-class CSV of each condition is written too.
- Needs `dataset/derived/phone/annotations.coco.json` (`python -m iiss.data.phone`).
- Can be stopped and started again: finished predictions are reused, scoring always runs again. `OVERWRITE = True` predicts again. A failing job does not stop the others; check the `status` column.
- APs/APm/APl use the `area` stored in the GT. Roboflow stores the bounding-box area.
- Close other GPU and memory-heavy programs. Phone photos are 12 MP, so 100 masks per image need more than 1 GB of RAM.
- `SMOKE = True`: S=320 checkpoints, 8 images per set, output in `runs/smoke/`.
- Keep the outputs after the real run: they are process evidence.

## Parameters

In [ ]:
MODELS = ['yolo', 'maskrcnn', 'm2f']
SIZES = [320, 640, 960]
EVAL_SETS = ['valid', 'test', 'phone']
SEED = 42
OVERWRITE = False     # True: export predictions again even if they are complete
SMOKE = False         # True: forces SIZES = [320]

In [ ]:
import pandas as pd
import torch

from iiss.eval.batch import condition_table, list_jobs, run_jobs, status_table
from iiss.utils.log import setup_logging

setup_logging()
pd.set_option('display.max_columns', 30, 'display.max_rows', 100, 'display.width', 200)
jobs = list_jobs(MODELS, [320] if SMOKE else SIZES, EVAL_SETS, SEED)
print(len(jobs), 'jobs;', torch.__version__,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Status before

In [ ]:
status_table(jobs)

## Predict and score

Interrupting the kernel is safe: run the cell again to continue.

In [ ]:
table = run_jobs(jobs, smoke=SMOKE, overwrite=OVERWRITE)

## Result

Mask (segm) AP unless the column says `bbox`. Any `failed` status is listed first.

In [ ]:
failed = table[table['status'] != 'ok']
print(f'{len(failed)} of {len(table)} jobs failed')
display(failed[['run_id', 'eval_set', 'status']])
display(table[table['status'] == 'ok'].drop(columns=['status', 'out_dir'], errors='ignore'))

## Phone set per condition

In [ ]:
condition_table(jobs, smoke=SMOKE)

## Next step

Send back the number of failed jobs and the result table. The tables and domain-shift analysis are built from `results/eval/` afterwards.